In [ ]:
#!/usr/bin/env python3
"""
=============================================================================
BBC Web Scraper — Dataset bilanciato per classificazione Sport / Non-Sport
=============================================================================
Autore:  [Il tuo nome]
Corso:   NLP – Progetto universitario (Fase 1: Raccolta dati)
Scopo:   Raccogliere 1 000 articoli "sport" e 1 000 articoli "non_sport"
         dal sito bbc.com e salvarli in un CSV con colonne:
         title, text, label
=============================================================================
"""

import csv
import time
import random
import logging
from urllib.parse import urljoin, urlparse
from collections import deque

import requests
from bs4 import BeautifulSoup

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURAZIONE GLOBALE
# ─────────────────────────────────────────────────────────────────────────────

# Quanti articoli per classe vogliamo raccogliere
OBIETTIVO_PER_CLASSE = 1000

# File di output
OUTPUT_CSV = "bbc_dataset.csv"

# Pausa tra una richiesta e l'altra (in secondi) — range casuale per sembrare
# più "umani" e non sovraccaricare il server
PAUSA_MIN = 0.5
PAUSA_MAX = 1

# Header HTTP che simulano un browser reale (riduce il rischio di blocco)
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/125.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-GB,en;q=0.9",
}

# Timeout massimo per ogni richiesta HTTP (secondi)
TIMEOUT_RICHIESTA = 15

# Segmenti di URL che indicano contenuti NON testuali (video, live, ecc.)
SEGMENTI_DA_ESCLUDERE = {
    "/video/", "/live/", "/programmes/", "/av/",
    "/sounds/", "/weather/", "/bitesize/", "/food/recipes/",
    "/mediaaction/", "/worklife/", "/reel/", "/iplayer/",
    "/archive/", "/cbbc/", "/cbeebies/", "/teach/",
    "/mundo/",   # edizione in spagnolo, struttura diversa
}

# Domini esterni da ignorare durante il crawling
DOMINIO_CONSENTITO = "bbc.com"

# Seed URL: pagine di partenza per ciascuna classe
SEED_SPORT = [
    "https://www.bbc.com/sport",
    "https://www.bbc.com/sport/football",
    "https://www.bbc.com/sport/cricket",
    "https://www.bbc.com/sport/tennis",
    "https://www.bbc.com/sport/athletics",
    "https://www.bbc.com/sport/cycling",
    "https://www.bbc.com/sport/formula1",
    "https://www.bbc.com/sport/rugby-union",
    "https://www.bbc.com/sport/golf",
    "https://www.bbc.com/sport/swimming",
    "https://www.bbc.com/sport/boxing",
]

SEED_NON_SPORT = [
    "https://www.bbc.com/news",
    "https://www.bbc.com/news/world",
    "https://www.bbc.com/news/business",
    "https://www.bbc.com/news/technology",
    "https://www.bbc.com/news/science_and_environment",
    "https://www.bbc.com/news/entertainment_and_arts",
    "https://www.bbc.com/news/health",
    "https://www.bbc.com/news/education",
    "https://www.bbc.com/culture",
    "https://www.bbc.com/travel",
    "https://www.bbc.com/future",
]

# ─────────────────────────────────────────────────────────────────────────────
# LOGGING — per monitorare i progressi a terminale
# ─────────────────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)s]  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# =============================================================================
#  FUNZIONI DI UTILITÀ
# =============================================================================

def pausa_educata():
    """Attende un tempo casuale tra PAUSA_MIN e PAUSA_MAX secondi."""
    attesa = random.uniform(PAUSA_MIN, PAUSA_MAX)
    time.sleep(attesa)


def scarica_pagina(url: str) -> BeautifulSoup | None:
    """
    Scarica una pagina web e restituisce un oggetto BeautifulSoup.
    Restituisce None se la richiesta fallisce o il content-type non è HTML.
    """
    try:
        risposta = requests.get(url, headers=HEADERS, timeout=TIMEOUT_RICHIESTA)
        risposta.raise_for_status()

        # Accettiamo solo HTML
        content_type = risposta.headers.get("Content-Type", "")
        if "text/html" not in content_type:
            return None

        return BeautifulSoup(risposta.text, "html.parser")

    except requests.RequestException as errore:
        log.debug("Errore nel download di %s: %s", url, errore)
        return None


# =============================================================================
#  FUNZIONI DI FILTRAGGIO URL
# =============================================================================

def è_url_valido(url: str) -> bool:
    """
    Verifica che un URL sia:
    - interno a bbc.com
    - non contenga segmenti esclusi (video, live, ecc.)
    - abbia un path sufficientemente lungo (evita homepage generiche)
    """
    parsed = urlparse(url)

    # Deve essere HTTP(S) e appartenere al dominio BBC
    if parsed.scheme not in ("http", "https"):
        return False
    if DOMINIO_CONSENTITO not in parsed.hostname:
        return False

    percorso = parsed.path.lower()

    # Escludiamo i segmenti non testuali
    for segmento in SEGMENTI_DA_ESCLUDERE:
        if segmento in percorso:
            return False

    # Escludiamo link di servizio / troppo corti
    if len(percorso.strip("/")) < 5:
        return False

    # Escludiamo frammenti e query string pesanti (spesso non sono articoli)
    if parsed.fragment:
        return False

    return True


def è_url_articolo(url: str) -> bool:
    """
    Euristica per distinguere pagine-articolo da pagine-indice.
    Gli articoli BBC di solito contengono un segmento alfanumerico finale
    (es. /sport/football/articles/c1234abcde oppure /news/articles/xxxx).
    Cerchiamo pattern tipici.
    """
    percorso = urlparse(url).path.lower().rstrip("/")
    segmenti = percorso.split("/")

    # Gli articoli più recenti hanno il segmento "articles" o "article"
    if "articles" in segmenti or "article" in segmenti:
        return True

    # Formato vecchio: /news/world-europe-12345678  (ultimo segmento con cifre)
    if segmenti:
        ultimo = segmenti[-1]
        # Contiene almeno un gruppo di cifre e non è troppo corto
        if any(c.isdigit() for c in ultimo) and len(ultimo) > 8:
            return True

    return False


def classifica_url(url: str) -> str | None:
    """
    Restituisce 'sport' se l'URL appartiene alla sezione sport,
    'non_sport' se appartiene ad altre sezioni riconosciute,
    None se non è classificabile con certezza.
    """
    percorso = urlparse(url).path.lower()

    # ---- SPORT ----
    if percorso.startswith("/sport"):
        return "sport"

    # ---- NON-SPORT ----
    sezioni_non_sport = (
        "/news", "/culture", "/travel", "/future",
        "/business", "/innovation", "/worklife",
        "/technology", "/science", "/entertainment",
        "/health", "/education", "/world",
    )
    for sezione in sezioni_non_sport:
        if percorso.startswith(sezione):
            return "non_sport"

    return None


# =============================================================================
#  FUNZIONE DI ESTRAZIONE CONTENUTO ARTICOLO
# =============================================================================

def estrai_articolo(soup: BeautifulSoup) -> tuple[str, str] | None:
    """
    Dato un oggetto BeautifulSoup di una pagina-articolo BBC,
    estrae titolo (H1) e corpo del testo pulito.

    Restituisce (titolo, testo) oppure None se l'estrazione fallisce
    o il contenuto è troppo corto per essere utile.
    """

    # ── 1. TITOLO ──
    tag_h1 = soup.find("h1")
    if not tag_h1:
        return None
    titolo = tag_h1.get_text(strip=True)
    if len(titolo) < 10:
        return None

    # ── 2. CORPO DEL TESTO ──
    # La BBC usa tipicamente un tag <article> come contenitore principale.
    articolo = soup.find("article")
    if not articolo:
        return None

    # Rimuoviamo elementi indesiderati dentro <article>
    # (navigazione, figure, aside, form, footer, cookie banner, "leggi anche")
    tag_da_rimuovere = [
        "nav", "footer", "aside", "figure", "figcaption",
        "form", "button", "script", "style", "svg", "iframe",
        "video", "audio",
    ]
    for tag_name in tag_da_rimuovere:
        for elemento in articolo.find_all(tag_name):
            elemento.decompose()

    # Rimuoviamo div con attributi che suggeriscono "leggi anche" / promozioni
    for div in articolo.find_all("div"):
        # Added check for malformed div tags where attrs might be None
        if div.attrs is None:
            continue
        ruolo = div.get("role", "").lower()
        classi = " ".join(div.get("class", [])).lower()
        data_comp = div.get("data-component", "").lower()
        if any(kw in classi for kw in ("related", "promo", "billboard", "ad-slot",
                                        "share", "social", "newsletter", "banner",
                                        "cookie", "consent", "topic-list")):
            div.decompose()
        elif ruolo in ("navigation", "banner", "complementary"):
            div.decompose()
        elif data_comp in ("related-stories", "tag-list", "links-block"):
            div.decompose()

    # Raccogliamo solo i paragrafi <p> rimasti — è il testo vero dell'articolo
    paragrafi = articolo.find_all("p")
    frasi = []
    for p in paragrafi:
        testo_p = p.get_text(strip=True)
        # Ignoriamo paragrafi troppo corti (didascalie, date, ecc.)
        if len(testo_p) > 40:
            frasi.append(testo_p)

    testo_completo = "\n".join(frasi)

    # Se il corpo è troppo corto, probabilmente non è un articolo vero
    if len(testo_completo) < 200:
        return None

    return titolo, testo_completo


# =============================================================================
#  FUNZIONE DI ESTRAZIONE LINK DA UNA PAGINA
# =============================================================================

def estrai_link(soup: BeautifulSoup, url_base: str) -> list[str]:
    """
    Estrae tutti i link <a href="..."> dalla pagina, li rende assoluti
    e restituisce solo quelli che passano il filtro di validità.
    """
    link_trovati = []
    for tag_a in soup.find_all("a", href=True):
        href = tag_a["href"]
        url_assoluto = urljoin(url_base, href)
        # Rimuoviamo frammenti (#...) e parametri query per normalizzare
        parsed = urlparse(url_assoluto)
        url_pulito = f"{parsed.scheme}://{parsed.netloc}{parsed.path}"
        if è_url_valido(url_pulito):
            link_trovati.append(url_pulito)
    return link_trovati


# =============================================================================
#  CRAWLING: motore principale (BFS per classe)
# =============================================================================

def raccogli_articoli(
    seed_urls: list[str],
    etichetta_target: str,
    obiettivo: int,
    url_globali_visitati: set,
) -> list[dict]:
    """
    Effettua un crawling BFS (Breadth-First Search) a partire dai seed URL.
    Raccoglie fino a `obiettivo` articoli la cui classe corrisponde a
    `etichetta_target` ('sport' o 'non_sport').

    Parametri
    ---------
    seed_urls : list[str]
        URL di partenza (homepage di sezione).
    etichetta_target : str
        Classe desiderata: 'sport' o 'non_sport'.
    obiettivo : int
        Numero di articoli da raccogliere.
    url_globali_visitati : set
        Set condiviso tra le due classi per evitare duplicati globali.

    Ritorna
    -------
    list[dict]  — lista di dizionari con chiavi 'title', 'text', 'label'.
    """

    # Coda BFS: partiamo dai seed
    coda = deque(seed_urls)
    articoli_raccolti = []
    pagine_analizzate = 0

    log.info("▶ Inizio crawling per classe '%s' (obiettivo: %d)", etichetta_target, obiettivo)

    while coda and len(articoli_raccolti) < obiettivo:
        url_corrente = coda.popleft()

        # Saltiamo se già visitato (controllo duplicati)
        if url_corrente in url_globali_visitati:
            continue
        url_globali_visitati.add(url_corrente)

        # Rate limiting educato
        pausa_educata()

        # Scarichiamo la pagina
        soup = scarica_pagina(url_corrente)
        if soup is None:
            continue

        pagine_analizzate += 1

        # ── Se la pagina sembra un articolo, proviamo a estrarlo ──
        if è_url_articolo(url_corrente):
            classe = classifica_url(url_corrente)
            if classe == etichetta_target:
                risultato = estrai_articolo(soup)
                if risultato:
                    titolo, testo = risultato
                    articoli_raccolti.append({
                        "title": titolo,
                        "text": testo,
                        "label": etichetta_target,
                    })
                    if len(articoli_raccolti) % 50 == 0:
                        log.info(
                            "   [%s] %d / %d articoli raccolti  (pagine analizzate: %d)",
                            etichetta_target,
                            len(articoli_raccolti),
                            obiettivo,
                            pagine_analizzate,
                        )

        # ── Estraiamo i link per continuare il crawling ──
        nuovi_link = estrai_link(soup, url_corrente)
        for link in nuovi_link:
            if link not in url_globali_visitati:
                # Diamo priorità ai link che appartengono alla classe target
                classe_link = classifica_url(link)
                if classe_link == etichetta_target:
                    coda.append(link)

        # Log periodico sullo stato della coda
        if pagine_analizzate % 100 == 0:
            log.info(
                "   [%s] Pagine analizzate: %d | Coda: %d | Articoli: %d/%d",
                etichetta_target, pagine_analizzate, len(coda),
                len(articoli_raccolti), obiettivo,
            )

    log.info(
        "✔ Crawling '%s' terminato: %d articoli raccolti su %d pagine analizzate.",
        etichetta_target, len(articoli_raccolti), pagine_analizzate,
    )
    return articoli_raccolti


# =============================================================================
#  SALVATAGGIO SU FILE CSV
# =============================================================================

def salva_csv(articoli: list[dict], percorso_file: str):
    """
    Salva la lista di articoli in un file CSV con colonne:
    title, text, label.
    Usa la codifica UTF-8 con BOM per compatibilità con Excel.
    """
    with open(percorso_file, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=["title", "text", "label"])
        writer.writeheader()
        writer.writerows(articoli)
    log.info("💾 Dataset salvato in '%s' (%d righe).", percorso_file, len(articoli))


# =============================================================================
#  MAIN — PUNTO DI INGRESSO
# =============================================================================

def main():
    """Orchestrazione: lancia il crawling per entrambe le classi e salva il CSV."""

    log.info("=" * 65)
    log.info("  BBC Web Scraper — Dataset bilanciato Sport / Non-Sport")
    log.info("=" * 65)

    # Set globale per evitare duplicati tra le due classi
    url_visitati = set()

    # ── Fase 1: raccolta articoli SPORT ──
    articoli_sport = raccogli_articoli(
        seed_urls=SEED_SPORT,
        etichetta_target="sport",
        obiettivo=OBIETTIVO_PER_CLASSE,
        url_globali_visitati=url_visitati,
    )

    # ── Fase 2: raccolta articoli NON-SPORT ──
    articoli_non_sport = raccogli_articoli(
        seed_urls=SEED_NON_SPORT,
        etichetta_target="non_sport",
        obiettivo=OBIETTIVO_PER_CLASSE,
        url_globali_visitati=url_visitati,
    )

    # ── Fase 3: unione e salvataggio ──
    dataset_completo = articoli_sport + articoli_non_sport

    # Mescoliamo per evitare che il modello impari l'ordine
    random.shuffle(dataset_completo)

    salva_csv(dataset_completo, OUTPUT_CSV)

    # ── Riepilogo finale ──
    n_sport = sum(1 for a in dataset_completo if a["label"] == "sport")
    n_non = sum(1 for a in dataset_completo if a["label"] == "non_sport")
    log.info("-" * 65)
    log.info("  RIEPILOGO FINALE")
    log.info("  Sport:     %d articoli", n_sport)
    log.info("  Non-Sport: %d articoli", n_non)
    log.info("  Totale:    %d articoli", len(dataset_completo))
    log.info("-" * 65)


if __name__ == "__main__":
    main()

10:00:06  [INFO]  =================================================================
10:00:06  [INFO]    BBC Web Scraper — Dataset bilanciato Sport / Non-Sport
10:00:06  [INFO]  =================================================================
10:00:06  [INFO]  ▶ Inizio crawling per classe 'sport' (obiettivo: 1000)
10:03:31  [INFO]     [sport] Pagine analizzate: 100 | Coda: 3817 | Articoli: 45/1000
10:03:48  [INFO]     [sport] 50 / 1000 articoli raccolti  (pagine analizzate: 109)
10:06:22  [INFO]     [sport] 100 / 1000 articoli raccolti  (pagine analizzate: 186)
10:06:50  [INFO]     [sport] Pagine analizzate: 200 | Coda: 5968 | Articoli: 103/1000
10:08:35  [INFO]     [sport] 150 / 1000 articoli raccolti  (pagine analizzate: 254)
10:10:00  [INFO]     [sport] Pagine analizzate: 300 | Coda: 7656 | Articoli: 189/1000
10:10:22  [INFO]     [sport] 200 / 1000 articoli raccolti  (pagine analizzate: 311)
10:12:15  [INFO]     [sport] 250 / 1000 articoli raccolti  (pagine analizzate: 371)
10:13:12